In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/meadowsm/siggy-styles-data/Employees Clients.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/Products Sold Grouped by Category for 01_01_2025 - 10_01_2026.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/Services Sold Grouped by Category for 01_01_2025 - 01_01_2026.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/Services Sold Grouped by Category for 01_02_2026 - 10_01_2026.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/siggy_client_spending_2025.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/siggy_client_spending_2026.csv


In [2]:
# Siggy Styles — SQL Cleaning & Analysis
# Service & Retail totals, monthly + annual, client data for her only
# Dates: 01/2025–09/2025 and 01/2026–09/2026
# ============================================================
import os, sqlite3, pandas as pd

# Avoiding path issues
for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        print(os.path.join(dirname, f))



/kaggle/input/datasets/meadowsm/siggy-styles-data/Employees Clients.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/Products Sold Grouped by Category for 01_01_2025 - 10_01_2026.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/Services Sold Grouped by Category for 01_01_2025 - 01_01_2026.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/Services Sold Grouped by Category for 01_02_2026 - 10_01_2026.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/siggy_client_spending_2025.csv
/kaggle/input/datasets/meadowsm/siggy-styles-data/siggy_client_spending_2026.csv


In [3]:
import os, sqlite3
import pandas as pd

path = "/kaggle/input/datasets/meadowsm/siggy-styles-data/"
conn = sqlite3.connect("siggy.db")

STYLIST = "Sigourney Marrero"
WINDOW = [("2025-01-01", "2025-09-30"), ("2026-01-01", "2026-09-30")] 


In [4]:
# Clean raw files immediately (drop PII columns)

PII_COLS = ["Client", "Client Name", "First Name", "Last Name",
            "Mobile Phone", "Phone", "Email Address", "Email"]

def load_sanitized(fname, **kwargs):
    df = pd.read_csv(path + fname, **kwargs)
    return df.drop(columns=[c for c in PII_COLS if c in df.columns], errors="ignore")

print("complete")

complete


In [5]:
# Services
svc_2025 = load_sanitized("Services Sold Grouped by Category for 01_01_2025 - 01_01_2026.csv", skiprows=1)
svc_2026 = load_sanitized("Services Sold Grouped by Category for 01_02_2026 - 10_01_2026.csv", skiprows=1)
df_services = pd.concat([svc_2025, svc_2026], ignore_index=True)

# Retail
df_products = load_sanitized("Products Sold Grouped by Category for 01_01_2025 - 10_01_2026.csv", skiprows=1)

In [6]:
# Client spending (already cleaned)
s25 = load_sanitized("siggy_client_spending_2025.csv")
s26 = load_sanitized("siggy_client_spending_2026.csv")


# Stylist client master (cleaned)
clients_master = pd.read_csv(path + "Employees Clients.csv")

# Standardize Client ID (fix data type)

def clean_ids(df):
    df = df.copy()
    df["Client ID"] = pd.to_numeric(df["Client ID"], errors="coerce")
    df = df.dropna(subset=["Client ID"])
    df["Client ID"] = df["Client ID"].astype(int).astype(str)
    return df

s25 = clean_ids(s25)
s26 = clean_ids(s26)
clients_master = clean_ids(clients_master)

In [7]:
# Clean dates + currency on service/retail, filter to stylist

def prep_sales(df, kind):
    df = df.copy()
    df["Date Closed"] = pd.to_datetime(df["Date Closed"], errors="coerce")
    df = df.dropna(subset=["Date Closed"])
    df["Year"] = df["Date Closed"].dt.year
    df["Month"] = df["Date Closed"].dt.month
    df["Total"] = pd.to_numeric(
        df["Total"].astype(str).str.replace("$", "").str.replace(",", ""),
        errors="coerce"
    )
    # Only her sales
    df = df[df["Employee"] == STYLIST]
    # Only the two windows
    mask = pd.Series(False, index=df.index)
    for start, end in WINDOW:
        mask |= (df["Date Closed"] >= start) & (df["Date Closed"] <= end)
    df = df[mask]
    df["Type"] = kind
    return df

df_services = prep_sales(df_services, "Service")
df_products = prep_sales(df_products, "Retail")

In [8]:
#Filter client spending to ONLY her clients 
valid_ids = set(clients_master["Client ID"].unique())

def keep_her_clients(df):
    return df[df["Client ID"].isin(valid_ids)].copy()

s25_her = keep_her_clients(s25)
s26_her = keep_her_clients(s26)

In [9]:
# SQL
df_services.to_sql("services", conn, if_exists="replace", index=False)
df_products.to_sql("retail", conn, if_exists="replace", index=False)
s25_her.to_sql("clients25", conn, if_exists="replace", index=False)
s26_her.to_sql("clients26", conn, if_exists="replace", index=False)

print("Rows -> services:", len(df_services), "| retail:", len(df_products),
      "| clients25:", len(s25_her), "| clients26:", len(s26_her))

Rows -> services: 3832 | retail: 565 | clients25: 232 | clients26: 233


In [10]:
print("\n=== SERVICE ANNUAL TOTALS ===")
print(pd.read_sql("""
    SELECT Year, SUM(Total) AS service_total
    FROM services
    GROUP BY Year
    ORDER BY Year;
""", conn).to_string(index=False))

print("\n=== SERVICE MONTHLY TOTALS ===")
print(pd.read_sql("""
    SELECT Year, Month, SUM(Total) AS service_total
    FROM services
    GROUP BY Year, Month
    ORDER BY Year, Month;
""", conn).to_string(index=False))

print("\n=== RETAIL ANNUAL TOTALS ===")
print(pd.read_sql("""
    SELECT Year, SUM(Total) AS retail_total
    FROM retail
    GROUP BY Year
    ORDER BY Year;
""", conn).to_string(index=False))

print("\n=== RETAIL MONTHLY TOTALS ===")
print(pd.read_sql("""
    SELECT Year, Month, SUM(Total) AS retail_total
    FROM retail
    GROUP BY Year, Month
    ORDER BY Year, Month;
""", conn).to_string(index=False))

print("\n=== SERVICE + RETAIL BY YEAR ===")
print(pd.read_sql("""
    SELECT Year,
           SUM(CASE WHEN Type='Service' THEN Total ELSE 0 END) AS service_total,
           SUM(CASE WHEN Type='Retail'  THEN Total ELSE 0 END) AS retail_total,
           SUM(Total) AS combined_total
    FROM (
        SELECT Year, Total, Type FROM services
        UNION ALL
        SELECT Year, Total, Type FROM retail
    )
    GROUP BY Year
    ORDER BY Year;
""", conn).to_string(index=False))


=== SERVICE ANNUAL TOTALS ===
 Year  service_total
 2025       136563.0
 2026       135453.0

=== SERVICE MONTHLY TOTALS ===
 Year  Month  service_total
 2025      1        17713.0
 2025      2        18579.0
 2025      3        21242.0
 2025      4        16337.0
 2025      5        13056.0
 2025      6        14745.0
 2025      7        11949.0
 2025      8        11941.0
 2025      9        11001.0
 2026      1        18097.0
 2026      2        19082.0
 2026      3        19618.0
 2026      4        14907.0
 2026      5        13242.0
 2026      6        15055.0
 2026      7        11168.0
 2026      8        11772.0
 2026      9        12512.0

=== RETAIL ANNUAL TOTALS ===
 Year  retail_total
 2025      12125.85
 2026      12467.45

=== RETAIL MONTHLY TOTALS ===
 Year  Month  retail_total
 2025      1       1189.70
 2025      2       1838.70
 2025      3       1704.50
 2025      4       1193.50
 2025      5       1632.25
 2025      6       1432.95
 2025      7        716.00
 2025

In [11]:
# Export cleaned CSVs
df_services.to_csv("siggy_services_clean.csv", index=False)
df_products.to_csv("siggy_retail_clean.csv", index=False)
s25_her.to_csv("siggy_clients_2025_clean.csv", index=False)
s26_her.to_csv("siggy_clients_2026_clean.csv", index=False)
print("\nExported cleaned CSVs.")

# Just wanted to verify no PII anywhere
def check_pii(df, name):
    found = [c for c in df.columns if c in PII_COLS]
    print(f"{name}: {'CLEAN' if not found else 'PII FOUND -> ' + str(found)}")

print("\n=== PII CHECK ===")
check_pii(df_services, "services")
check_pii(df_products, "retail")
check_pii(s25_her, "clients25")
check_pii(s26_her, "clients26")
check_pii(clients_master, "clients_master")

print("\nScanning exported files for PII column names...")
for f in ["siggy_services_clean.csv", "siggy_retail_clean.csv",
          "siggy_clients_2025_clean.csv", "siggy_clients_2026_clean.csv"]:
    if os.path.exists(f):
        cols = pd.read_csv(f, nrows=1).columns.tolist()
        bad = [c for c in cols if c in PII_COLS]
        print(f"{f}: {'CLEAN' if not bad else 'PII -> ' + str(bad)}")

conn.close()
print("\nDone.")


Exported cleaned CSVs.

=== PII CHECK ===
services: CLEAN
retail: CLEAN
clients25: CLEAN
clients26: CLEAN
clients_master: CLEAN

Scanning exported files for PII column names...
siggy_services_clean.csv: CLEAN
siggy_retail_clean.csv: CLEAN
siggy_clients_2025_clean.csv: CLEAN
siggy_clients_2026_clean.csv: CLEAN

Done.
